# Lab 3 — one controlled change and a defensible recommendation

100 points · 6% NP · same pairs as Lab 2 · due Thu 29 October 2026, 12:30 Santiago.

Lab 1 checked data and simple rules. Lab 2 changed one input under fixed comparison conditions.
Here you keep the inputs fixed and change one model setting, then explain errors and write for a
Head of Strategy. The supplied code runs the experiment; you must supply the checked explanation,
one-change specification, three supported failure patterns, memo and truthful AI record.

Follow the numbered headings. See README for setup and GLOSSARY for terms.
A losing additional candidate can earn full credit. The unchanged starter is incomplete.


## Environment and data check

Extract the whole ZIP and choose your kernel before running this block. Follow README.md if a check fails.

In [1]:
from pathlib import Path
import sys
import importlib
print('Selected Python:', sys.executable)
print('Version:', sys.version.split()[0], '| working directory:', Path.cwd())
required_files = ['lab03_support_v3.py', 'pitwindow_student_v2.py', 'data/learning_v2.csv']
missing = [name for name in required_files if not Path(name).is_file()]
if missing:
    raise FileNotFoundError('Extract the complete ZIP and open its folder as cwd. Missing: ' + ', '.join(missing))
for module in ['numpy', 'pandas', 'sklearn', 'scipy', 'joblib']:
    loaded = importlib.import_module(module)
    print(module, loaded.__version__)
print('Ready: local files/imports found. Next cell checks data integrity and split.')


Selected Python: /home/martin/Documents/U/Trimestre 2026_3/Artificial Intelligence Workshop/Lab2_Benjamin_Martin/.venv/bin/python
Version: 3.13.5 | working directory: /home/martin/Documents/U/Trimestre 2026_3/Artificial Intelligence Workshop/IIT414W_Lab03_StudentPack_v3
numpy 2.5.3
pandas 3.0.6
sklearn 1.9.1
scipy 1.18.1
joblib 1.6.0
Ready: local files/imports found. Next cell checks data integrity and split.


In [2]:
from pathlib import Path
import pandas as pd
from lab03_support_v3 import (
    FEATURES, SEED, BUDGET, load_learning, run_base_comparison,
    run_one_change, export_run, markdown_table,
)
ROOT = Path('.')
train, dev = load_learning(ROOT)
print('Seed:', SEED, '| development alert budget:', BUDGET)
print('Permitted inputs:', FEATURES)
audit = pd.DataFrame([
    {'split': name, 'rows': len(df), 'positive_windows': int(df.y.sum()),
     'eligible_events': df.loc[df.y.eq(1), 'event_id'].nunique(),
     'first_year': int(df.year.min()), 'last_year': int(df.year.max())}
    for name, df in [('train', train), ('dev', dev)]
])
display(audit)
assert set(train.opportunity_id).isdisjoint(dev.opportunity_id)


Seed: 414 | development alert budget: 0.1
Permitted inputs: ['circuit_id', 'compound', 'tyre_life', 'lap_number', 'progress', 'prior_pit_entries', 'pace_median_s', 'pace_slope_s_per_lap', 'pace_n']


,split,rows,positive_windows,eligible_events,first_year,last_year
0,train,63165,5523,1934,2018,2021
1,dev,9451,1073,370,2022,2022


## 1. Define the prediction

**Prediction time.** Each row (`opportunity_id`) is one opportunity at the end of lap L for a rival driver; the cut-off time is `cut_s`. Using only what is known at that moment, we predict whether **that same driver** enters the pits during **their own next three laps** (up to `horizon_end_s`).

**Positive label.** `y = 1` if an observed pit entry falls inside that horizon. It is an **observed** rival entry, not a recommendation of when our own car should stop. One entry (`event_id`) usually produces several positive windows: in dev there are 1073 positive windows belonging to 370 distinct entries (≈2.9 windows per entry).

**One permitted input.** `tyre_life`: laps completed on the current tyre, known at the end of lap L. It is one of the nine `FEATURES` and the most important input for both forests. Caveat: it is the last reported value and can lag.

**One future-known field kept out of X.** `event_s` (and also `event_id`) describes when or which pit entry happened, which is only known *after* the cut-off. Using it as an input would be leakage: the answer would be in the inputs. The same applies to `horizon_end_s` and `future_neutralisation` (whether a safety car/VSC happened later). `event_id` is used only to audit events in the metrics.

**Limits of the snapshot.** This is a corrected historical dataset (FastF1, manifest v2) and may contain revisions made after the race. It supports comparing models under the same protocol. It does **not** show that each value would have arrived in time during a live race, it says nothing about real feed latency, and it is not an independent prospective trial.

## Practice reading the counts

Illustrative manual example only, NOT F1 data. Two positive windows refer to event A; a third refers
to event B. The final window is negative and triggers a false alert.

| Window | y | Event | Alert |
|---|---:|---|---|
| w1 | 1 | A | yes |
| w2 | 1 | A | yes |
| w3 | 1 | B | no |
| w4 | 0 | none | yes |

Ungraded check: count TP, FP, FN, alerts, eligible/caught events, precision and event recall.
What changes if w2 stops alerting? Think first, then open the self-check below.


### Optional practice response

Before opening the self-check below, write or think through: TP = __; FP = __; FN = __; alerts = __; eligible/caught events = __ / __; precision = __; event recall = __. If w2 becomes quiet: __. This exercise is ungraded; no separate submission is required.

<details>
<summary>Open after trying: self-check for the four fictional windows</summary>

TP=2, FP=1, FN=1, TN=0; alerts=3; precision=2/3. There are two eligible events A/B, one caught, so event recall=1/2. If w2 becomes quiet: TP=1, FP=1, FN=2, alerts=2, precision=1/2; A is still caught by w1 and event recall remains 1/2.

</details>

In [3]:
base_comparison, fitted, base_policies = run_base_comparison(train, dev)
compact_columns = ['model', 'alerts', 'precision', 'events_caught', 'eligible_events', 'event_recall']
print('Start with DEVELOPMENT. Undefined precision means there were no alerts.')
display(base_comparison.loc[base_comparison.split.eq('dev'), compact_columns])
columns = ['model', 'split', 'n', 'TP', 'FP', 'FN', 'TN', 'alerts', 'alert_rate',
           'precision', 'events_caught', 'eligible_events', 'event_recall', 'threshold']
print("Audit view: train and development reuse the threshold selected for that model on dev.")
display(base_comparison[columns])


Start with DEVELOPMENT. Undefined precision means there were no alerts.


,model,alerts,precision,events_caught,eligible_events,event_recall
1,never_alert,0,NaN,0,370,0.000000
3,historical_baseline,931,0.268528,94,370,0.254054
5,tree,571,0.269702,63,370,0.170270
7,forest_all,938,0.276119,107,370,0.289189
9,forest_sqrt,945,0.359788,140,370,0.378378


Audit view: train and development reuse the threshold selected for that model on dev.


,model,split,n,TP,FP,FN,TN,alerts,alert_rate,precision,events_caught,eligible_events,event_recall,threshold
0,never_alert,train,63165,0,0,5523,57642,0,0.000000,NaN,0,1934,0.000000,NaN
1,never_alert,dev,9451,0,0,1073,8378,0,0.000000,NaN,0,370,0.000000,NaN
2,historical_baseline,train,63165,3042,6645,2481,50997,9687,0.153360,0.314029,1137,1934,0.587901,0.161926
3,historical_baseline,dev,9451,250,681,823,7697,931,0.098508,0.268528,94,370,0.254054,0.161926
4,tree,train,63165,1816,4831,3707,52811,6647,0.105232,0.273206,650,1934,0.336091,0.713735
5,tree,dev,9451,154,417,919,7961,571,0.060417,0.269702,63,370,0.170270,0.713735
6,forest_all,train,63165,2503,4503,3020,53139,7006,0.110916,0.357265,969,1934,0.501034,0.718176
7,forest_all,dev,9451,259,679,814,7699,938,0.099249,0.276119,107,370,0.289189,0.718176
8,forest_sqrt,train,63165,2671,6414,2852,51228,9085,0.143830,0.294001,1005,1934,0.519648,0.608283
9,forest_sqrt,dev,9451,340,605,733,7773,945,0.099989,0.359788,140,370,0.378378,0.608283


## 2. Explain the common comparison

**Base development results (2022, rounds 1–11; 9451 windows, 1073 positive, 370 eligible entries; limit 945 alerts = 10%).**

| system | alerts | precision (TP/alerts) | entries detected / eligible |
|---|---:|---:|---:|
| never_alert | 0 | undefined | 0/370 |
| historical_baseline | 931 | 250/931 = 0.269 | 94/370 = 0.254 |
| tree | 571 | 154/571 = 0.270 | 63/370 = 0.170 |
| forest_all | 938 | 259/938 = 0.276 | 107/370 = 0.289 |
| forest_sqrt | 945 | 340/945 = 0.360 | 140/370 = 0.378 |

**forest_all vs forest_sqrt.** With almost the same number of alerts (938 vs 945), `forest_sqrt` detects 33 more entries (140 vs 107) and its precision rises from 0.276 to 0.360. Both use 150 trees, `max_depth=6` and `min_samples_leaf=30`; only `max_features` differs. After preprocessing there are 48 columns: 7 numeric, 3 missing-value indicators, and the one-hot columns for `circuit_id` and `compound`. With `max_features=1.0` every split considers all 48; with `'sqrt'` every split draws ⌊√48⌋ = 6 random candidates. No column is removed: it is available again at the next split or in another tree. Plausible mechanism (not measured here): in `forest_all` the trees differ only in their bootstrap sample, so they tend to repeat the same dominant splits (`tyre_life`, `progress`). In `forest_sqrt` the restriction forces some trees to use other inputs (`prior_pit_entries`, `lap_number`), and the average ranks 2022 cases better. We did not measure correlation between trees; this is a hypothesis consistent with the numbers.

**Undefined precision with zero alerts.** `never_alert` issues no alerts, so precision would be 0/0: it is undefined (not 0). Its event recall is 0/370.

**TP windows ≠ detected entries.** An entry counts as detected if **at least one** of its positive windows is alerted. `forest_sqrt` has 340 TP windows but 140 detected entries, because several TP windows can belong to the same entry.

**Same policy, different thresholds.** Every system uses the same rule: at most 10% alerts on dev, maximise distinct entries detected, with precision and threshold as tie-breakers. The resulting number differs because each model scores on its own scale: 0.608 for forest_sqrt, 0.718 for forest_all, 0.714 for tree and 0.162 for the historical baseline. The tree stops at 571 alerts (6.0%) even though 945 are allowed: it has only 8 leaves and therefore only 8 distinct scores on dev. Moving to the next leaf would add all its tied windows and reach 1898 alerts, over the limit, so the rule stops earlier. We compare *model + alert policy*, not the model alone.

**One train/dev gap.** `historical_baseline` detects 1137/1934 = 0.588 of entries on train but 94/370 = 0.254 on dev, with the same threshold of 0.162. It alerts on 15.3% of windows on train and 9.9% on dev. Train figures are *in-sample*: the rates per circuit × compound × tyre-age band were computed on those same rows. Dev figures were used to choose the threshold, so neither is an untouched final test. Possible causes (hypotheses, not shown): overfitting to small train cells, the 2022 regulation and tyre change (dev contains only SOFT/MEDIUM/HARD plus wet compounds), or both. The gap alone does not prove its cause. The scores are also not calibrated probabilities; for example, the models use `class_weight='balanced'`.

## 3. Record your expectation before your additional run

The base outputs are already supplied. Complete `experiment_spec.md` before running YOUR candidate
for the first time. If already run, say so and keep a retrospective account. Do not invent history.

Choose ONE tuple: tree / max_depth / 6 (old 3); forest_sqrt / max_depth / 3 (old 6);
or forest_sqrt / min_samples_leaf / 100 (old 30). No search for the best of all three.
The helper clones the base pipeline and changes one setting. Its threshold is reselected on dev
under the same policy, so the numeric threshold may differ. This compares the resulting alert systems.


Choose one additional configuration. Repeating that same configuration for debugging or Restart + Run All is allowed. Disclose any different configurations you already tried; do not hide a search.

In [4]:
EXPERIMENT_MODEL_ID = 'tree'        # 'tree' or 'forest_sqrt', matching the allowed table
EXPERIMENT_PARAMETER = 'max_depth'  # 'max_depth' or 'min_samples_leaf'
EXPERIMENT_VALUE = 6                # the one allowed new value (parent: 3)


In [5]:
change = None
policies = dict(base_policies)
comparison = base_comparison.copy()
if any(value is None for value in [EXPERIMENT_MODEL_ID, EXPERIMENT_PARAMETER, EXPERIMENT_VALUE]):
    print('INCOMPLETE STARTER: configure one additional experiment in extra-config.')
else:
    extra_rows, extra_model, extra_policy, change = run_one_change(
        train, dev, fitted, EXPERIMENT_MODEL_ID, EXPERIMENT_PARAMETER, EXPERIMENT_VALUE)
    comparison = pd.concat([base_comparison, extra_rows], ignore_index=True)
    policies[change['model_id']] = extra_policy
    display(comparison.loc[comparison.model.isin([change['parent'], change['model_id']]), columns])


,model,split,n,TP,FP,FN,TN,alerts,alert_rate,precision,events_caught,eligible_events,event_recall,threshold
4,tree,train,63165,1816,4831,3707,52811,6647,0.105232,0.273206,650,1934,0.336091,0.713735
5,tree,dev,9451,154,417,919,7961,571,0.060417,0.269702,63,370,0.170270,0.713735
10,tree__max_depth_6,train,63165,2068,3741,3455,53901,5809,0.091965,0.355999,775,1934,0.400724,0.790333
11,tree__max_depth_6,dev,9451,145,435,928,7943,580,0.061369,0.250000,57,370,0.154054,0.790333


## 4. Explain your additional result

**Candidate:** `tree__max_depth_6`, a clone of `tree` in which only `max_depth` changes from 3 to 6 (`changed_parameters = ['model__max_depth']`). Its threshold was re-selected on dev with the same rule.

| development (9451 windows, 370 entries) | tree (parent) | tree__max_depth_6 |
|---|---:|---:|
| alerts (rate) | 571 (6.0%) | 580 (6.1%) |
| TP / FP | 154 / 417 | 145 / 435 |
| precision TP/alerts | 154/571 = 0.270 | 145/580 = 0.250 |
| entries detected / eligible | 63/370 = 0.170 | 57/370 = 0.154 |
| threshold selected on dev | 0.714 | 0.790 |
| train: entries detected / eligible | 650/1934 = 0.336 | 775/1934 = 0.401 |

**Did the expectation hold?** Mostly not. We expected the extra depth to give finer scores, letting the threshold approach the 945 allowed alerts and detect more entries. Observed: alerts barely changed (571 → 580) and detected entries **fell** from 63 to 57 of 370. Three other expectations held: it does not beat `forest_sqrt` (140/370), precision fell slightly (0.270 → 0.250), and the train/dev gap grew. For entry detection the parent goes from 0.336 on train to 0.170 on dev (gap 0.166); the candidate goes from 0.401 to 0.154 (gap 0.247). It improves on train and gets worse on dev.

**What we observed about the mechanism.** The tree went from 8 to 58 leaves, but only 32 distinct scores appear on dev, and large tied blocks remain. The top 8 score levels together contain 580 windows. The next level (score 0.736) alone contains 794 dev windows: including it would give 1374 alerts, over the 945 limit, so the rule stops again at ~6%. Our premise was wrong: more leaves do not guarantee small blocks *at the top of the ranking*. Also, the highest-scoring leaf (0.942) contains 51 dev windows and **none** is positive. All 51 come from 11 drivers in race `2022-06-12`, on HARD tyres, laps 39–47 (e.g. `2022-06-12:1:46:8888.147000`).

**Likely cause (hypothesis, not measured).** The extra splits, with leaves of ≥50 rows, seem to capture 2018–2021 combinations (circuit × compound × tyre age) that do not recur the same way in 2022. This is consistent with the gain on train (+125 detected entries, precision 0.273 → 0.356) and the loss on dev. We did not separate how much is due to the model and how much to the new threshold: we compare the whole system, model + policy. Dev AP rises slightly (0.180 → 0.200), suggesting a somewhat better overall ranking, but not in the top 6–10% where alerts are issued.

**Bounded recommendation.** Do not adopt `tree__max_depth_6`. Within this comparison, the best development system remains `forest_sqrt`: 140/370 entries with 945 alerts, precision 0.360. Limits: dev has only 11 races of 2022 and was used to choose the thresholds, so all these figures are optimistic and do not replace an untouched final test. The data are a historical snapshot that does not prove live availability, and the scores are not calibrated probabilities. The `2022-06-12` case is a single race: it is worth checking (section 5 returns to it), not evidence of a general effect.

## 5. Explain three distinct failure patterns

Keep your additional candidate and its selected threshold frozen. Inspect the supplied tables, choose the denominator for your claim, and use the queries to find supporting cases. This step does not train or retune.

### Choose your denominator

| Question within your context | Calculation | Denominator counts |
|---|---|---|
| Of the alerts issued, how many were false? | FP / (TP + FP) | Alerted windows; false-alert fraction, equal to 1 − precision when defined. |
| Of the negative windows, how many alerted? | FP / (FP + TN) | Negative windows; false positive rate (FPR). |
| Of the positive windows, how many were quiet? | FN / (TP + FN) | Positive windows; `miss_rate` in the context export. |
| Of the eligible events, how many were completely missed? | Completely missed events / eligible events | Distinct pit entries; use all positive windows for each event. |

A zero denominator means undefined, shown as NaN; it does not mean zero errors. Name your population and unit with every fraction. Window context groups do not define an event population automatically: if describing missed events in a context, state which distinct events you include and check all their positive windows, including those outside that context. The event query below checks a whole event.


### A complete entry from the four fictional windows

This uses only the four-window practice example, not the Rival dataset or an additional candidate's results.

“For the context of all four toy windows, w4 is a false alert (y=0, alert=yes). One of the three alerts is false: FP/alerts=1/3. The traceable case is `opportunity_id=w4`. This could prompt an unnecessary review of a rival that does not enter within the horizon; next check the time and outcome around w4 before attributing a cause. With only four invented windows, this illustrates the reasoning format and cannot establish a general effect.”

The same counts give FPR=1/(1+0)=1/1 over negative windows, and miss_rate=1/(2+1)=1/3 over positive windows. Those fractions answer different questions. Event B has one positive window (w3), none alerted: one of two eligible events is completely missed. A quiet w2 alone would not make event A missed because w1 still alerts.


In [6]:
from lab03_support_v3 import development_predictions, context_error_summary, event_error_summary
predictions = None
if change is None:
    print('INCOMPLETE: run your additional candidate before examining its errors.')
else:
    predictions = development_predictions(dev, extra_model, extra_policy, change['model_id'])
    context_errors = context_error_summary(predictions)
    event_errors = event_error_summary(predictions)
    print('Descriptive context tables. No refit or threshold change occurs here.')
    display(context_errors)
    print('FP window examples (highest scores first):')
    display(predictions.loc[predictions.error_type.eq('FP')].sort_values('score', ascending=False).head(8))
    print('FN window examples (lowest scores first):')
    display(predictions.loc[predictions.error_type.eq('FN')].sort_values('score').head(8))
    print('Missed events:', int(event_errors.missed_event.sum()), '/', len(event_errors))
    display(event_errors.loc[event_errors.missed_event].head(8))


Descriptive context tables. No refit or threshold change occurs here.


,dimension,context,n,positive_windows,alerts,TP,FP,FN,TN,precision,miss_rate
0,compound,HARD,4037,266,394,60,334,206,3437,0.152284,0.774436
1,compound,INTERMEDIATE,377,86,0,0,0,86,291,NaN,1.000000
2,compound,MEDIUM,3593,473,116,54,62,419,3058,0.465517,0.885835
3,compound,SOFT,1211,201,70,31,39,170,971,0.442857,0.845771
4,compound,UNKNOWN,4,0,0,0,0,0,4,NaN,NaN
5,compound,WET,229,47,0,0,0,47,182,NaN,1.000000
6,circuit_id,10,797,61,60,10,50,51,686,0.166667,0.836066
7,circuit_id,144,720,88,60,9,51,79,581,0.150000,0.897727
8,circuit_id,149,617,55,106,10,96,45,466,0.094340,0.818182
9,circuit_id,15,1032,159,4,4,0,155,873,1.000000,0.974843


FP window examples (highest scores first):


,opportunity_id,race_id,driver_id,lap_number,circuit_id,compound,tyre_life,progress,event_id,y,model,score,alert,error_type
6651,2022-06-12:63:45:8821.572000,2022-06-12,63,45,144,HARD,12.0,0.882353,NaN,0.0,tree__max_depth_6,0.941621,True,FP
6608,2022-06-12:6:43:8743.205000,2022-06-12,6,43,144,HARD,25.0,0.843137,NaN,0.0,tree__max_depth_6,0.941621,True,FP
6653,2022-06-12:63:47:9036.809000,2022-06-12,63,47,144,HARD,14.0,0.921569,NaN,0.0,tree__max_depth_6,0.941621,True,FP
6607,2022-06-12:6:42:8631.343000,2022-06-12,6,42,144,HARD,24.0,0.823529,NaN,0.0,tree__max_depth_6,0.941621,True,FP
6652,2022-06-12:63:46:8929.253000,2022-06-12,63,46,144,HARD,13.0,0.901961,NaN,0.0,tree__max_depth_6,0.941621,True,FP
6566,2022-06-12:5:44:8745.538000,2022-06-12,5,44,144,HARD,35.0,0.862745,NaN,0.0,tree__max_depth_6,0.941621,True,FP
6567,2022-06-12:5:45:8854.709000,2022-06-12,5,45,144,HARD,36.0,0.882353,NaN,0.0,tree__max_depth_6,0.941621,True,FP
6568,2022-06-12:5:46:8963.417000,2022-06-12,5,46,144,HARD,37.0,0.901961,NaN,0.0,tree__max_depth_6,0.941621,True,FP


FN window examples (lowest scores first):


,opportunity_id,race_id,driver_id,lap_number,circuit_id,compound,tyre_life,progress,event_id,y,model,score,alert,error_type
3164,2022-04-24:77:16:5381.689000,2022-04-24,77,16,6,INTERMEDIATE,15.0,0.253968,2022-04-24:77:pit:5568.070000,1.0,tree__max_depth_6,0.0,False,FN
6495,2022-06-12:47:8:4688.207000,2022-06-12,47,8,144,HARD,7.0,0.156863,2022-06-12:47:pit:4823.868000,1.0,tree__max_depth_6,0.0,False,FN
6494,2022-06-12:47:7:4576.311000,2022-06-12,47,7,144,HARD,6.0,0.137255,2022-06-12:47:pit:4823.868000,1.0,tree__max_depth_6,0.0,False,FN
3110,2022-04-24:63:16:5380.085000,2022-04-24,63,16,6,INTERMEDIATE,15.0,0.253968,2022-04-24:63:pit:5566.538000,1.0,tree__max_depth_6,0.0,False,FN
3111,2022-04-24:63:17:5473.705000,2022-04-24,63,17,6,INTERMEDIATE,16.0,0.269841,2022-04-24:63:pit:5566.538000,1.0,tree__max_depth_6,0.0,False,FN
3163,2022-04-24:77:15:5288.329000,2022-04-24,77,15,6,INTERMEDIATE,14.0,0.238095,2022-04-24:77:pit:5568.070000,1.0,tree__max_depth_6,0.0,False,FN
6493,2022-06-12:47:6:4464.779000,2022-06-12,47,6,144,HARD,5.0,0.117647,2022-06-12:47:pit:4823.868000,1.0,tree__max_depth_6,0.0,False,FN
2475,2022-04-24:18:17:5492.368000,2022-04-24,18,17,6,INTERMEDIATE,16.0,0.269841,2022-04-24:18:pit:5587.137000,1.0,tree__max_depth_6,0.0,False,FN


Missed events: 313 / 370


,event_id,race_id,driver_id,positive_windows,alerted_positive_windows,caught,missed_event
0,2022-03-20:10:pit:5169.339000,2022-03-20,10,3,0,False,True
2,2022-03-20:11:pit:5253.086000,2022-03-20,11,3,0,False,True
4,2022-03-20:11:pit:8037.416000,2022-03-20,11,3,0,False,True
5,2022-03-20:14:pit:4871.171000,2022-03-20,14,3,0,False,True
8,2022-03-20:16:pit:5239.419000,2022-03-20,16,3,0,False,True
10,2022-03-20:16:pit:8333.486000,2022-03-20,16,3,0,False,True
13,2022-03-20:18:pit:8372.487000,2022-03-20,18,2,0,False,True
14,2022-03-20:1:pit:5143.479000,2022-03-20,1,3,0,False,True


### Find the cases behind your claim

Set CONTEXT_COLUMN to a supplied context column (for example compound or circuit_id), CONTEXT_VALUE to an actual value of the same type shown in your predictions, and ERROR_KIND to FP or FN. The summary uses all rows in that context, so its denominators do not accidentally count only errors. `cases` contains every matching case; the display previews eight. For three patterns you can repeat the query with recorded settings, or copy this query block within this section. Keep each setting and relevant output alongside its written claim.

For a whole-event claim, copy an event_id from the missed-event table into EVENT_TO_INSPECT. The second query retrieves all its positive windows, including their opportunity_id values, without a context filter. An event is missed only if none alerts. Unknown IDs return no rows and are not evidence of a missed event.


In [7]:
from lab03_support_v3 import context_query, positive_windows_for_event

CONTEXT_COLUMN = 'compound'
CONTEXT_VALUE = 'HARD'
ERROR_KIND = 'FP'

if predictions is None:
    print('Run your additional candidate first; its frozen predictions are needed.')
elif CONTEXT_COLUMN is None or CONTEXT_VALUE is None:
    print('Set the context column/value above to inspect your own error evidence.')
else:
    context_summary, cases = context_query(predictions, CONTEXT_COLUMN, CONTEXT_VALUE, ERROR_KIND)
    display(context_summary)
    print('Matching cases:', len(cases), '(first eight shown)')
    display(cases.head(8))


,context_column,context_value,n,TP,FP,FN,TN,alerts,negative_windows,positive_windows,false_alert_fraction,false_positive_rate,miss_rate
0,compound,HARD,4037,60,334,206,3437,394,3771,266,0.847716,0.088571,0.774436


Matching cases: 334 (first eight shown)


,opportunity_id,race_id,driver_id,lap_number,circuit_id,compound,tyre_life,progress,event_id,y,model,score,alert,error_type
151,2022-03-20:14:38:7605.497000,2022-03-20,14,38,63,HARD,12.0,0.666667,NaN,0.0,tree__max_depth_6,0.852841,True,FP
575,2022-03-20:4:37:7542.135000,2022-03-20,4,37,63,HARD,12.0,0.649123,NaN,0.0,tree__max_depth_6,0.852841,True,FP
603,2022-03-20:44:23:6066.726000,2022-03-20,44,23,63,HARD,12.0,0.403509,NaN,0.0,tree__max_depth_6,0.852841,True,FP
776,2022-03-20:63:27:6468.947000,2022-03-20,63,27,63,HARD,12.0,0.473684,NaN,0.0,tree__max_depth_6,0.852841,True,FP
777,2022-03-20:63:28:6568.244000,2022-03-20,63,28,63,HARD,13.0,0.491228,NaN,0.0,tree__max_depth_6,0.852841,True,FP
778,2022-03-20:63:29:6667.518000,2022-03-20,63,29,63,HARD,14.0,0.508772,NaN,0.0,tree__max_depth_6,0.852841,True,FP
857,2022-03-27:1:24:6311.946000,2022-03-27,1,24,149,HARD,7.0,0.480000,NaN,0.0,tree__max_depth_6,0.890659,True,FP
858,2022-03-27:1:25:6405.247000,2022-03-27,1,25,149,HARD,8.0,0.500000,NaN,0.0,tree__max_depth_6,0.890659,True,FP


In [8]:
EVENT_TO_INSPECT = '2022-04-24:10:pit:5498.635000'

if predictions is None:
    print('Run your additional candidate first.')
elif EVENT_TO_INSPECT is None:
    print('Set an event_id above when checking a completely missed event.')
else:
    event_windows = positive_windows_for_event(predictions, EVENT_TO_INSPECT)
    display(event_windows)
    if event_windows.empty:
        print('No positive windows found: check the ID; do not call this a missed event.')
    else:
        print('Positive windows:', len(event_windows), 'Alerted:', int(event_windows.alert.sum()))
        print('Completely missed event:', not bool(event_windows.alert.any()))


,opportunity_id,race_id,driver_id,lap_number,circuit_id,compound,tyre_life,progress,event_id,y,model,score,alert,error_type
2315,2022-04-24:10:14:5213.979000,2022-04-24,10,14,6,INTERMEDIATE,13.0,0.222222,2022-04-24:10:pit:5498.635000,1.0,tree__max_depth_6,0.0,False,FN
2316,2022-04-24:10:15:5308.891000,2022-04-24,10,15,6,INTERMEDIATE,14.0,0.238095,2022-04-24:10:pit:5498.635000,1.0,tree__max_depth_6,0.0,False,FN
2317,2022-04-24:10:16:5404.064000,2022-04-24,10,16,6,INTERMEDIATE,15.0,0.253968,2022-04-24:10:pit:5498.635000,1.0,tree__max_depth_6,0.0,False,FN


Positive windows: 3 Alerted: 0
Completely missed event: True


### What counts as a separate pattern

The same error type in different contexts can count as separate patterns when each entry states its context rule, gives its own count/denominator and traceable case, and explains a consequence and next check. Three cases of the same error in the same context count as one pattern. Changing only a context name without evidence and interpretation is incomplete. You do not need three different causes, independent groups or statistical significance. Declare overlap; do not add overlapping counts as distinct errors.

Fictional examples for interpreting the requirement (these are not Rival results):

- Acceptable entry A: “For context `compound == TOY_A`, 2 of 6 alerts are false (FP/alerts), including `toy-A-4`. These alerts require checking a rival that does not enter within the horizon; next check the neighbouring windows to see whether alerts repeat around the same situation. The six-alert sample is small.”
- Acceptable entry B: “For context `compound == TOY_B`, 3 of 10 alerts are false, including `toy-B-2`. The review burden also occurs here; next inspect the recorded timing around those cases before proposing a cause.” With their separate supporting rows, A and B count as two patterns even though both are FP. Your submission must provide the actual rows behind its own claims.
- Incomplete: “TOY_A has false alerts; TOY_B has false alerts; TOY_C has false alerts.” Context names alone supply neither counts/denominators nor cases, consequences or checks. Three IDs from TOY_A under the same rule also describe one pattern.


### Your three entries


All counts come from the frozen development predictions of `tree__max_depth_6` (threshold 0.790, 580 alerts, 57/370 entries caught). Nothing was refit or re-thresholded.

1. **Pattern / rule: false alerts on HARD tyres** (`compound == 'HARD'`, error FP, window level).
   **Evidence:** 334 of 394 HARD alerts are false (FP/alerts = 0.848), compared with 62/116 = 0.534 on MEDIUM and 39/70 = 0.557 on SOFT. Over HARD negative windows, FPR = 334/3771 = 0.089. Observed: 66 of the 74 HARD alerts with `progress ≥ 0.7` are FP, and the top-scoring leaf (score 0.942) holds 51 dev windows from race `2022-06-12`, all negative.
   **opportunity_id:** `2022-06-12:14:42:8534.626000` (HARD, lap 42, tyre_life 23, score 0.942, y = 0).
   **Consequence / next check:** most HARD alerts would send the strategist to watch a rival who does not pit within three laps, which erodes trust in the alert. Hypothesis (not measured): the tree links high tyre age on HARD to a stop, while in 2022 many drivers ran the HARD to the flag. Next check: for these FP windows, look up whether the driver made any later stop in that race (i.e. whether they were already on their final stint).

2. **Pattern / rule: completely missed entries on wet-weather tyres** (`compound` in INTERMEDIATE/WET, error FN, checked at event level).
   **Evidence:** 0 alerts in the 606 INTERMEDIATE/WET windows; 133 of 133 positive windows are quiet (86 INTERMEDIATE + 47 WET, miss_rate = 1.0). At event level, 46 eligible entries have positive windows on wet tyres. All of their positive windows are on wet tyres, and 46/46 are completely missed (none of their positive windows alerted). That is 46 of the 313 missed entries. The highest wet-tyre score is 0.736, below the 0.790 threshold.
   **opportunity_id:** `2022-04-24:10:14:5213.979000`, a positive window of event `2022-04-24:10:pit:5498.635000` (INTERMEDIATE, score 0.0). `positive_windows_for_event` shows 0 of its 3 positive windows alerted.
   **Consequence / next check:** the alert stays silent exactly when conditions are changing and stop decisions cluster (wet ↔ slick crossovers), so it cannot be used in mixed-weather races. Hypothesis (not measured): wet-tyre rows are rare in 2018–2021 training data, and weather-driven stops depend on information the nine inputs do not contain. Next check: count train rows and positive rate for INTERMEDIATE/WET, and compare with the parent tree on the same windows.

3. **Pattern / rule: almost every entry missed in one dry race** (`circuit_id == '15'`, race `2022-05-22`, error FN, checked at event level).
   **Evidence:** 4 alerts in 1032 windows; 155 of 159 positive windows are quiet (miss_rate = 0.975). At event level, 51 of the 53 eligible entries in this race are completely missed. All positive windows here are on SOFT or MEDIUM, so this pattern does not overlap with pattern 2.
   **opportunity_id:** `2022-05-22:10:7:4372.266000`, a positive window of event `2022-05-22:10:pit:4647.680000` (SOFT, lap 7, tyre_life 6, score 0.511). `positive_windows_for_event` shows 0 of its 3 positive windows alerted.
   **Consequence / next check:** in a race with many stops, the strategist would get almost no warning, so the tool gives a false sense that rivals are not about to pit. We checked one simple explanation: the median tyre_life of positive windows here is 16 laps, versus 14 in the other dry races, so "earlier stops than usual" is **not** supported. The cause remains unknown. Next check: compare this circuit's train-period (2018–2021) positive rate and scores by lap with 2022 to see whether the stop pattern changed.

**Overlap:** pattern 1 counts FP windows, while patterns 2 and 3 count FN windows and missed events. Patterns 2 and 3 cover different races and disjoint events (46 + 51 = 97 of the 313 missed entries), so they can be added. The other 216 missed entries are not explained by these three patterns. Each context is a single season of 11 races; pattern 3 is one race, so it shows something to investigate, not a general effect.

In [9]:
manifest = export_run(comparison, policies, change, ROOT, predictions=predictions)
print('Additional experiment executed:', manifest['experiment_complete'])
print('This flag does NOT check your reasoning, three patterns, memo or AI record.')
print('Systems:', comparison.model.nunique(), '| train/dev rows:', len(comparison))
print('Generated comparison_table.md and all five outputs/ files. No manual transcription needed.')


Additional experiment executed: True
This flag does NOT check your reasoning, three patterns, memo or AI record.
Systems: 6 | train/dev rows: 12
Generated comparison_table.md and all five outputs/ files. No manual transcription needed.


## 6. Memo and submission

Complete memo.md and PROMPTS.md. The comparison table is generated automatically.
Check CHECKLIST.md, restart + run all, save visible outputs and push the final commit.
Completed comparison: six systems / twelve train-dev rows, plus your additional candidate's diagnostics.
Canvas text entry: both names, accessible repository URL, final commit and notebook path.
The execution flag does not certify your explanation or completed documents.


## Use your feedback next time

Ungraded preparation: revisit one relevant Lab 2 comment, or identify a weakness yourself if no
comment applies. What did you check differently here? Add one or two sentences if useful.
After grading, carry the specific next action from your Lab 3 feedback into the capstone experiment plan.
No extra file, points or oral interview is added by this reflection.
